# Synthetic camera RAW 50 MP Colab

Incremental implementation of PRD_v3. Output will be a **synthetic sensor reconstruction of a hypothetical
exposure**, never a genuine camera capture. The early RAW gate exports tiny test DNGs only; full reconstruction and TIFF export remain pending.
Metadata must carry D/I/F/M provenance; DNG Make must be `Synthetic`. No Sony MakerNotes,
realistic Sony serials, date inference or GPS inference are permitted.

Run in Colab only. Configure the panel before running Input audit. `FAST_DEV_RUN` reduces planned
sensor dimensions; this foundation only decodes the source and computes geometry arithmetically.
No weights or calibration assets are downloaded. Later compute stages must benchmark initial tiles,
checkpoint to host RAM/disk and release GPU tensors between stages.


## Environment + reproducibility

In [ ]:
# Colab setup: replace this branch fallback with a reviewed 40-character commit SHA
# for a reproducible release. REPO_REF is the commit SHA placeholder until then.
import re
import subprocess
import sys

REPO_REF = "fm/exp50mp-v3-synthetic-raw"
if REPO_REF != "fm/exp50mp-v3-synthetic-raw" and not re.fullmatch(r"[0-9a-fA-F]{40}", REPO_REF):
    raise ValueError("Use the documented branch fallback or a full commit SHA")
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        f"git+https://github.com/RDC28/experiment_50MP.git@{REPO_REF}",
    ]
)
# This repository distribution installs the top-level synthraw package.

### Configuration

Every field comes from `ExperimentConfig`. Edit controls, then run Input audit to validate and freeze
settings. PATH uses a runtime path. DRIVE mounts Drive and uses INPUT_PATH inside that mount.
CALIBRATION_ASSETS accepts a JSON list of runtime paths. Leave INPUT_PATH empty for UPLOAD.
If ipywidgets is unavailable, edit `CONFIG_VALUES` in this cell and rerun it. Changing configuration
requires rerunning Input audit and subsequent cells to start a fresh run.


In [ ]:
import json
from dataclasses import fields
from typing import Literal, get_args, get_origin, get_type_hints

from IPython.display import display

from synthraw.config import ExperimentConfig

CONFIG_VALUES = ExperimentConfig().to_dict()
controls = {}
try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

if widgets is None:
    print("ipywidgets unavailable: edit CONFIG_VALUES above, then run Input audit.")
    print(json.dumps(CONFIG_VALUES, indent=2))
else:
    hints = get_type_hints(ExperimentConfig)
    for field in fields(ExperimentConfig):
        name, value, hint = field.name, CONFIG_VALUES[field.name], hints[field.name]
        options = dict(description=name, style={"description_width": "260px"}, layout=widgets.Layout(width="100%"))
        if get_origin(hint) is Literal:
            control = widgets.Dropdown(options=get_args(hint), value=value, **options)
        elif hint is bool:
            control = widgets.Checkbox(value=value, **options)
        elif hint is int:
            # Text avoids imposing widget integer limits on NumPy seed values.
            control = widgets.Text(value=str(value), **options)
        elif hint is float:
            control = widgets.FloatText(value=value, **options)
        else:
            text = json.dumps(value) if name == "CALIBRATION_ASSETS" else (value or "")
            control = widgets.Text(value=text, **options)
        controls[name] = control
    display(widgets.VBox(list(controls.values())))


def read_configuration():
    """Validate current controls, including dependent choices, before any input I/O."""
    if not controls:
        return ExperimentConfig.from_dict(CONFIG_VALUES)
    values = {name: control.value for name, control in controls.items()}
    for field in fields(ExperimentConfig):
        if get_type_hints(ExperimentConfig)[field.name] is int:
            values[field.name] = int(values[field.name])
    values["INPUT_PATH"] = values["INPUT_PATH"] or None
    values["CALIBRATION_ASSETS"] = json.loads(values["CALIBRATION_ASSETS"])
    return ExperimentConfig.from_dict(values)

In [ ]:
from synthraw.environment import EnvironmentInspector

environment = EnvironmentInspector().inspect()
EnvironmentInspector.print_report(environment)

### Mandatory tiny RAW gate

Run this before Input audit or any reconstruction. Eight 64 x 48 Bayer fixtures test all CFA
phases at both ADC depths, including the experimental virtual 16-bit profile. Each DNG must
reopen with identical pixels and matching metadata through independent LibRaw and develop to
finite 16-bit RGB. Any failure stops execution; fix the runtime before continuing. Temporary
DNGs are removed on success or failure. These fixtures are synthetic test data, not the input image.

In [ ]:
# Mandatory RAW gate: only 64 x 48 fixtures, before any reconstruction.
from dataclasses import replace
from tempfile import TemporaryDirectory

import numpy as np

from synthraw.dng_metadata import CFA_PATTERNS, DNGMetadataBuilder
from synthraw.dng_writer import DNGWriter
from synthraw.libraw_validator import LibRawValidator

RAW_GATE_PASSED = False


def run_raw_gate():
    """Fail immediately if independent LibRaw decoding or development fails."""
    from pathlib import Path

    from synthraw.geometry import TargetGeometryPlanner

    tiny_geometry = replace(
        TargetGeometryPlanner().plan(64, 48),
        width=64,
        height=48,
        active_area=(0, 0, 48, 64),
        default_crop_origin=(0, 0),
        default_crop_size=(64, 48),
    )
    with TemporaryDirectory(prefix="synthraw_gate_") as directory:
        for bits in (14, 16):
            gate_config = ExperimentConfig(ADC_BITS=bits)
            for phase in CFA_PATTERNS:
                records = DNGMetadataBuilder().build(
                    tiny_geometry,
                    gate_config,
                    cfa_phase=phase,
                    as_shot_neutral=(0.5, 1, 0.75),
                    color_matrix2=((0.8, 0.1, 0.05), (0.05, 1, 0.1), (0.02, 0.1, 0.9)),
                )
                mosaic = np.arange(48 * 64, dtype=np.uint16).reshape(48, 64) + 512
                path = DNGWriter().write(Path(directory) / f"{phase}_{bits}.dng", mosaic, records, gate_config)
                LibRawValidator().validate(path, mosaic, records, develop=True)


run_raw_gate()
RAW_GATE_PASSED = True
print("RAW gate passed: all four Bayer phases at 14 and virtual 16 bits; LibRaw pixels, metadata and RGB verified.")

## Input audit

In [ ]:
if not globals().get("RAW_GATE_PASSED", False):
    raise RuntimeError("Run the mandatory tiny RAW gate before Input audit")

import shutil
from pathlib import Path

from synthraw.input_audit import InputInspector
from synthraw.run_layout import RandomStreams, RunLayout

config = read_configuration()
if config.INPUT_MODE == "UPLOAD":
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one JPEG or PNG")
    source_path = Path(next(iter(uploaded)))
    del uploaded  # Colab already saved the bytes to source_path.
else:
    if config.INPUT_MODE == "DRIVE":
        from google.colab import drive

        drive.mount(config.DRIVE_MOUNT_PATH)
    source_path = Path(config.INPUT_PATH).expanduser()
    if config.INPUT_MODE == "DRIVE" and not source_path.resolve().is_relative_to(
        Path(config.DRIVE_MOUNT_PATH).resolve()
    ):
        raise ValueError("DRIVE INPUT_PATH must be inside DRIVE_MOUNT_PATH")

audit = InputInspector().inspect(source_path)
run = RunLayout.create("/content/synthraw_runs")
streams = RandomStreams.from_seeds(config.SCENE_SEED, config.SENSOR_NOISE_SEED, config.SENSOR_BODY_SEED)
shutil.copy2(source_path, run.directory("source") / source_path.name)
(run.directory("metadata") / "config.json").write_text(json.dumps(config.to_dict(), indent=2), encoding="utf-8")
(run.directory("metadata") / "input_audit.json").write_text(json.dumps(audit.to_dict(), indent=2), encoding="utf-8")
print(json.dumps(audit.to_dict(), indent=2))
print(f"Run directory: {run.root}")

## Profile selection

In [ ]:
print(f"Target profile: {config.TARGET_PROFILE}")
print(f"Profile source: {config.CAMERA_PROFILE_SOURCE}; spectral mode: {config.SPECTRAL_MODE}")
# RAW milestone: SonyMetadataRegistry and CameraProfileManager will validate profile assets.

## Target geometry

In [ ]:
from dataclasses import asdict

from synthraw.geometry import TargetGeometryPlanner

geometry = TargetGeometryPlanner().plan(audit.width, audit.height, config)
geometry_json = json.dumps(asdict(geometry), indent=2)
(run.directory("metadata") / "geometry_plan.json").write_text(geometry_json, encoding="utf-8")
print(geometry_json)
print("Geometry and memory estimates only; no full sensor arrays allocated.")

## Source diagnostics

In [ ]:
print(f"Source: {audit.resolution_class}, aspect {audit.aspect_ratio:.6f}")
print(f"Near-black RGB fractions: {audit.near_black_fractions}")
print(f"Near-white RGB fractions: {audit.near_white_fractions}")
# Reconstruction milestone: add spatial detail, ringing and compression diagnostics.

## SR reconstruction

In [ ]:
# Reconstruction milestone placeholder:
# SuperResolutionEngine: tiled SR, pinned lazy checkpoints, seam checks and tile ETA.
print("Pending reconstruction milestone.")

## HDR / scene-linear reconstruction

In [ ]:
# Reconstruction milestone placeholder:
# HDRReconstructionEngine: scene-linear reconstruction and confidence masks.
print("Pending reconstruction milestone.")

## Camera geometry + exposure inference

In [ ]:
# Camera milestone placeholder:
# CameraGeometryEstimator and ExposurePosteriorEstimator: geometry and exposure posterior.
print("Pending camera milestone.")

## Inverse ISP / camera colour

In [ ]:
# Camera milestone placeholder:
# InverseISPEngine and CameraColourModel: inverse tone processing and colour matrices.
print("Pending camera milestone.")

## Optical forward model

In [ ]:
# Camera milestone placeholder:
# LensSimulator: optional optical forward model.
print("Pending camera milestone.")

## Sensor/electron simulation

In [ ]:
# Sensor milestone placeholder:
# ElectronSignalSimulator and SensorNoiseSimulator: independent seeded sensor realisations.
print("Pending sensor milestone.")

## CFA construction

In [ ]:
# Sensor milestone placeholder:
# CFAMosaicer: Bayer CFA construction.
print("Pending sensor milestone.")

## ADC quantisation

In [ ]:
# Sensor milestone placeholder:
# ADCSimulator: 14-bit or explicitly virtual 16-bit quantisation.
print("Pending sensor milestone.")

## Metadata construction

In [ ]:
# RAW milestone placeholder:
# DNGMetadataBuilder and SonyARWEmulationBuilder: D/I/F/M field provenance.
print("Pending raw milestone.")

## DNG export

In [ ]:
# RAW milestone placeholder:
# DNGWriter: trivial synthetic gradient first, then LibRaw round trip.
print("Pending raw milestone.")

## RAW round-trip development

In [ ]:
# Validation milestone placeholder:
# LibRawValidator and DemosaicValidator: independent RAW development.
print("Pending validation milestone.")

## 16-bit TIFF export

In [ ]:
# Integration milestone placeholder:
# TIFFWriter: 16-bit master export.
print("Pending integration milestone.")

## Validation suite

In [ ]:
# Validation milestone placeholder:
# NoiseValidator, ColourValidator and EditabilityValidator: scientific checks.
print("Pending validation milestone.")

## 2K/4K comparison

In [ ]:
# Validation milestone placeholder:
# SourceResolutionComparator: 2K/4K experiment.
print("Pending validation milestone.")

## Provenance + report packaging

In [ ]:
# Integration milestone placeholder:
# ProvenanceWriter and ReportGenerator: metadata records and report packaging.
print("Pending integration milestone.")